SQL Analysis

Objective:

Queries cover segment revenue analysis, customer lifetime value, monthly trends, channel behaviour, and churn prediction cross-referencing.

1 . Imports

In [3]:
# importing libraries 
import os
import sqlite3
import pandas as pd
print("Libraries imported successfully")

# ── Path configuration ────────────────────────────────────────────────────────
BASE_DIR = os.path.expanduser('~/Downloads/Customer Life cycle Analysis')

print(f'Project root : {BASE_DIR}')
print(f'Data folder  : {os.path.join(BASE_DIR, "Data")}')
print(f'Folders exist: data={os.path.exists(os.path.join(BASE_DIR, "Data"))}, outputs={os.path.exists(os.path.join(BASE_DIR, "Outputs"))}')
print('Libraries loaded ✓')

Libraries imported successfully
Project root : /Users/umaobbani/Downloads/Customer Life cycle Analysis
Data folder  : /Users/umaobbani/Downloads/Customer Life cycle Analysis/Data
Folders exist: data=True, outputs=False
Libraries loaded ✓


2. Connecting the database

Loading both CSV outputs into a local SQLite database.
SQLite runs entirely inside Python — no server or installation required.
The .db file is created in the /data/ folder alongside the CSVs.

In [11]:
# ── Connect to database ───────────────────────────────────────────────────────
DB_PATH = os.path.join(BASE_DIR, 'Data', 'customer_lifecycle.db')
print(f'Connecting to database at: {DB_PATH}')
# ── Connect to database ───────────────────────────────────────────────────────
conn    = sqlite3.connect(DB_PATH)

Connecting to database at: /Users/umaobbani/Downloads/Customer Life cycle Analysis/Data/customer_lifecycle.db


In [9]:
# load CSV data into pandas DataFrame
rfm_df = pd.read_csv(os.path.join(BASE_DIR, 'Data', 'rfm_segments.csv'),dtype={'CustomerID': str})
print(f'Loaded RFM segments data with shape: {rfm_df.shape}')

df = pd.read_csv(os.path.join(BASE_DIR, 'Data', 'cleaned_online_retail_data.csv'), dtype={'Customer ID': str}, parse_dates=['InvoiceDate'])
print(f'Loaded cleaned online retail data with shape: {df.shape}')


Loaded RFM segments data with shape: (5881, 10)
Loaded cleaned online retail data with shape: (1033036, 13)


In [10]:
df.head(5)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalRevenue,InvoiceYearMonth,InvoiceYear,DayOfWeek,Hour
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4,2009-12,2009,Tuesday,7
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009-12,2009,Tuesday,7
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009-12,2009,Tuesday,7
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8,2009-12,2009,Tuesday,7
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,2009-12,2009,Tuesday,7


In [12]:
# ── Write to database ─────────────────────────────────────────────────────────
rfm_df.to_sql('rfm_segments', conn, if_exists='replace', index=False)
df.to_sql('transactions',    conn, if_exists='replace', index=False)

1033036

In [15]:
print(f'Database     : {DB_PATH}')
print(f'rfm_segments : {len(rfm_df):,} rows')
print(f'transactions : {len(df):,} rows')
print(f'Total rows   : {len(rfm_df) + len(df):,}')

Database     : /Users/umaobbani/Downloads/Customer Life cycle Analysis/Data/customer_lifecycle.db
rfm_segments : 5,881 rows
transactions : 1,033,036 rows
Total rows   : 1,038,917


2. Query Helper Function

A reusable function that takes a SQL string, runs it against the database, and prints clean results.
Calling it with a semicolon ; suppresses the duplicate DataFrame display in Jupyter.



In [20]:
def execute_query(sql, description=None):
    result = pd.read_sql_query(sql, conn)
    if description:
        print(f"\n{description}\n")
    print(result.to_string(index=False))
    print(f"\nQuery executed successfully. Result shape: {result.shape}\n")
    return result
print("Function 'execute_query' defined successfully.")


Function 'execute_query' defined successfully.


3. Executing Queries

Query 1 — Segment Revenue Summary

GROUP BY with multiple aggregations. Baseline summary confirming segment sizes and revenue from the RFM analysis.

In [21]:
execute_query("""
              SELECT * FROM rfm_segments LIMIT 5
              """, description="Sample RFM Segments Data")
              
              
              
              


Sample RFM Segments Data

 Customer ID  Recency  Frequency  Monetary  R_Score  F_Score  M_Score  RFM_Score RFM_Label         Segment
     12346.0     5716         12  77556.46        2        5        5       4.00     2-5-5  Need Attention
     12347.0     5393          8   4921.53        5        4        5       4.67     5-4-5       Champions
     12348.0     5466          5   2019.40        3        4        4       3.67     3-4-4 Loyal Customers
     12349.0     5409          4   4428.69        5        3        5       4.33     5-3-5 Loyal Customers
     12350.0     5701          1    334.40        2        1        2       1.67     2-1-2            Lost

Query executed successfully. Result shape: (5, 10)



,Customer ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,RFM_Label,Segment
0,12346.0,5716,12,77556.46,2,5,5,4.00,2-5-5,Need Attention
1,12347.0,5393,8,4921.53,5,4,5,4.67,5-4-5,Champions
2,12348.0,5466,5,2019.40,3,4,4,3.67,3-4-4,Loyal Customers
3,12349.0,5409,4,4428.69,5,3,5,4.33,5-3-5,Loyal Customers
4,12350.0,5701,1,334.40,2,1,2,1.67,2-1-2,Lost


In [25]:
execute_query("""
            SELECT
             segment as segment,
             COUNT(*) as customer_count,
             SUM(monetary) as total_revenue,
             AVG(monetary) as avg_revenue_per_customer
            FROM rfm_segments
            GROUP BY segment        
            ORDER BY total_revenue DESC    
              
              """, description="Segments Revenue Summary")




Segments Revenue Summary

        segment  customer_count  total_revenue  avg_revenue_per_customer
      Champions            1476    12001386.39               8131.020589
Loyal Customers            1231     2536528.60               2060.543136
 Need Attention             550     1132093.50               2058.351818
           Lost            1524      653884.67                429.058182
Potential Loyal             829      596645.47                719.717093
        At Risk              88      264052.31               3000.594432
  New Customers             183      190213.31               1039.416995

Query executed successfully. Result shape: (7, 4)



,segment,customer_count,total_revenue,avg_revenue_per_customer
0,Champions,1476,12001386.39,8131.020589
1,Loyal Customers,1231,2536528.60,2060.543136
2,Need Attention,550,1132093.50,2058.351818
3,Lost,1524,653884.67,429.058182
4,Potential Loyal,829,596645.47,719.717093
5,At Risk,88,264052.31,3000.594432
6,New Customers,183,190213.31,1039.416995


Query 2 — Revenue Concentration (Window Function)

Uses SUM() OVER () — a window function that calculates the grand total across all rows simultaneously, enabling percentage calculation without a subquery.

In [ ]:
execute_query("""
              SELECT 
              segment as segment,
              SUM(monetary) as total_revenue,
              SUM(monetary)*100/ SUM(SUM(monetary)) OVER()  as revenue_percentage
              FROM rfm_segments
              GROUP BY segment
              ORDER BY total_revenue DESC   
              """, description="Segments Revenue Contribution")
              


Segments Revenue Contribution

        segment  total_revenue  revenue_percentage
      Champions    12001386.39           69.073506
Loyal Customers     2536528.60           14.598890
 Need Attention     1132093.50            6.515719
           Lost      653884.67            3.763407
Potential Loyal      596645.47            3.433969
        At Risk      264052.31            1.519743
  New Customers      190213.31            1.094765

Query executed successfully. Result shape: (7, 3)



,segment,total_revenue,revenue_percentage
0,Champions,12001386.39,69.073506
1,Loyal Customers,2536528.60,14.598890
2,Need Attention,1132093.50,6.515719
3,Lost,653884.67,3.763407
4,Potential Loyal,596645.47,3.433969
5,At Risk,264052.31,1.519743
6,New Customers,190213.31,1.094765


Query 3 — Top 10 Customers by Lifetime Value

In [54]:
execute_query("""
    SELECT
        "Customer ID",
        Frequency,
        Recency,
        SUM(Monetary) AS total_revenue
     
    FROM rfm_segments
    GROUP BY "Customer ID", Frequency, Recency
    ORDER BY total_revenue DESC
    LIMIT 10
""", description="Top 10 Customers by Lifetime Value")



Top 10 Customers by Lifetime Value

 Customer ID  Frequency  Recency  total_revenue
     18102.0        145     5391      580987.04
     14646.0        152     5392      528602.52
     14156.0        156     5400      313437.62
     14911.0        398     5392      291420.81
     17450.0         51     5399      244784.25
     13694.0        143     5394      195640.69
     17511.0         60     5393      172132.87
     16446.0          2     5391      168472.50
     16684.0         55     5395      147142.77
     12415.0         28     5415      144458.37

Query executed successfully. Result shape: (10, 4)



,Customer ID,Frequency,Recency,total_revenue
0,18102.0,145,5391,580987.04
1,14646.0,152,5392,528602.52
2,14156.0,156,5400,313437.62
3,14911.0,398,5392,291420.81
4,17450.0,51,5399,244784.25
5,13694.0,143,5394,195640.69
6,17511.0,60,5393,172132.87
7,16446.0,2,5391,168472.50
8,16684.0,55,5395,147142.77
9,12415.0,28,5415,144458.37


Query 4 — Monthly Revenue Trend (JOIN)

INNER JOIN between transactions and rfm_segments on Customer ID. Confirms the Q4 seasonality pattern found in Notebook 01 using SQL.

In [60]:
execute_query("""
    SELECT
        t."InvoiceYearMonth" AS invoice_yearmonth,
        COUNT(DISTINCT t."Invoice") AS total_invoices,
        COUNT(DISTINCT t."Customer ID") AS total_customers,
        SUM(t."TotalRevenue") AS total_revenue
    FROM transactions t
    INNER JOIN rfm_segments r
        ON t."Customer ID" = r."Customer ID"
    GROUP BY t."InvoiceYearMonth"
    ORDER BY t."InvoiceYearMonth"
""", description="Monthly Revenue Trend")



Monthly Revenue Trend

invoice_yearmonth  total_invoices  total_customers  total_revenue
          2009-12            1884             1030     661085.650
          2010-01            1284              774     530875.822
          2010-02            1335              807     487596.426
          2010-03            1896             1100     642216.951
          2010-04            1607              991     571889.712
          2010-05            1767             1061     557887.490
          2010-06            1827             1089     603723.670
          2010-07            1713              988     560885.330
          2010-08            1547              964     585259.460
          2010-09            2039             1200     778662.711
          2010-10            2579             1570     963365.050
          2010-11            3144             1682    1129054.012
          2010-12            1707              947     552374.110
          2011-01            1236              783  

,invoice_yearmonth,total_invoices,total_customers,total_revenue
0,2009-12,1884,1030,661085.650
1,2010-01,1284,774,530875.822
2,2010-02,1335,807,487596.426
3,2010-03,1896,1100,642216.951
4,2010-04,1607,991,571889.712
5,2010-05,1767,1061,557887.490
6,2010-06,1827,1089,603723.670
7,2010-07,1713,988,560885.330
8,2010-08,1547,964,585259.460
9,2010-09,2039,1200,778662.711
